In [ ]:
import pandas as pd
import numpy as np
import os

def read_netlist(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
    netlist = []
    for line in lines:
        parts = line.strip().replace('(', '').replace(')', '').split()
        netlist.append(parts)
    return netlist

def read_ports(filename):
    with open(filename, 'r') as file:
        ports = file.readline().strip().split()
    return ports

def build_connection_matrix(netlist, ports):
    # DEBUGGING: Print initial netlist and ports
    print(f"Netlist length: {len(netlist)}")
    print(f"Ports: {ports}")

    # Component to number mapping
    component_mapping = {
        'resistor': 1,
        'capacitor': 2,
        'inductor': 3,
        'diode': 4,
        # 'nmos4': 5,
        # 'pmos4': 6,
        'npn': 7,
        'pnp': 8
    }
    
    # MODIFICATION START: More robust filtering
    # Explicitly check for empty netlist or ports
    if not netlist or not ports:
        print("Empty netlist or ports")
        return None, None

    # Check if the circuit contains inverter, XOR, or PFD components
    forbidden_components = ['INVERTER', 'XOR', 'PFD', 'nmos4', 'pmos4']
    for component in netlist:
        if component[-1].upper() in forbidden_components:
            print(f"Skipping due to forbidden component: {component[-1]}")
            return None, None
    
    # Check if netlist exceeds 5 lines
    if len(netlist) > 5:
        print(f"Skipping due to netlist size: {len(netlist)} lines")
        return None, None
    
    # Create component list with numerical representations
    component_list = []
    for component in netlist:
        component_type = component[-1].lower()
        if component_type in component_mapping:
            component_list.append(component_mapping[component_type])
        else:
            print(f"Unrecognized component type: {component_type}")
            return None, None
    
    # Initialize node list with ports
    nodes = ports[:]
    
    # Counters for each component type
    counters = {
        # 'pmos4': 1,
        # 'nmos4': 1,
        'npn': 1,
        'pnp': 1,
        'resistor': 1,
        'capacitor': 1,
        'inductor': 1,
        'diode': 1
    }
    
    # Define node types based on the component type
    node_types = {
        # 'pmos4': lambda i: [f'PM{i}', f'PM{i}_D', f'PM{i}_G', f'PM{i}_S', f'PM{i}_B'],
        # 'nmos4': lambda i: [f'NM{i}', f'NM{i}_D', f'NM{i}_G', f'NM{i}_S', f'NM{i}_B'],
        'npn': lambda i: [f'NPN{i}', f'NPN{i}_C', f'NPN{i}_B', f'NPN{i}_E'],
        'pnp': lambda i: [f'PNP{i}', f'PNP{i}_C', f'PNP{i}_B', f'PNP{i}_E'],
        'resistor': lambda i: [f'R{i}', f'R{i}_P', f'R{i}_N'],
        'capacitor': lambda i: [f'C{i}', f'C{i}_P', f'C{i}_N'],
        'inductor': lambda i: [f'L{i}', f'L{i}_P', f'L{i}_N'],
        'diode': lambda i: [f'DIO{i}', f'DIO{i}_P', f'DIO{i}_N']
    }
    
    # Add component-specific nodes
    for component in netlist:
        component_type = component[-1]
        index = counters[component_type]
        counters[component_type] += 1
        
        if component_type in node_types:
            nodes_to_add = node_types[component_type](index)
            nodes.extend(nodes_to_add)
    
    # Create an empty connection matrix
    matrix = pd.DataFrame(0, index=nodes, columns=nodes)
    
    # Fill the matrix with connections based on the netlist
    net_connections = []
    new_counters = {k: 1 for k in counters.keys()}
    
    for component in netlist:
        element = component[1:-1]
        component_type = component[-1]
        index = new_counters[component_type]
        new_counters[component_type] += 1
        
        # Determine connection nodes based on component type
        # if component_type == 'nmos4':
        #     connections = [f'NM{index}_D', f'NM{index}_G', f'NM{index}_S', f'NM{index}_B']
        # elif component_type == 'pmos4':
        #     connections = [f'PM{index}_D', f'PM{index}_G', f'PM{index}_S', f'PM{index}_B']
        if component_type == 'npn':
            connections = [f'NPN{index}_C', f'NPN{index}_B', f'NPN{index}_E']
        elif component_type == 'pnp':
            connections = [f'PNP{index}_C', f'PNP{index}_B', f'PNP{index}_E']
        elif component_type == 'resistor':
            connections = [f'R{index}_P', f'R{index}_N']
        elif component_type == 'capacitor':
            connections = [f'C{index}_P', f'C{index}_N']
        elif component_type == 'inductor':
            connections = [f'L{index}_P', f'L{index}_N']
        elif component_type == 'diode':
            connections = [f'DIO{index}_P', f'DIO{index}_N']
        else:
            continue
        
        # Create connections
        for conn, el in zip(connections, element):
            if el in nodes:
                matrix.at[conn, el] = 1
                matrix.at[el, conn] = 1
            else:
                net_connections.append((conn, el))
    
    # Create net connections
    net_dict = {}
    for conn, net in net_connections:
        if net not in net_dict:
            net_dict[net] = []
        net_dict[net].append(conn)
    
    # Fill matrix with indirect connections
    for net, conn_list in net_dict.items():
        for i in range(len(conn_list)):
            for j in range(i + 1, len(conn_list)):
                matrix.at[conn_list[i], conn_list[j]] = 1
                matrix.at[conn_list[j], conn_list[i]] = 1
    
    return matrix, component_list

start = 1
end = 3502
# Add a counter for successful and skipped circuits
successful_circuits = 0
skipped_circuits = 0

for i in range(start, end+1):
    print(f"Processing circuit {i}")
    number = str(i)
    # Define file names
    netlist_file = 'Dataset/' + number + '/' + number + '.cir'
    port_file =  'Dataset/' + number + '/' + 'Port' + number + '.txt' 
    
    # Additional error handling and skipping
    if not os.path.isfile(netlist_file):
        print(f"Directory '{netlist_file}' does not exist. Skipping...")
        skipped_circuits += 1
        continue

    # Read netlist and ports
    try:
        netlist = read_netlist(netlist_file)
        ports = read_ports(port_file)
    except Exception as e:
        print(f"Error reading files for circuit {number}: {e}")
        skipped_circuits += 1
        continue

    # Build the connection matrix
    result = build_connection_matrix(netlist, ports)

    # Skip if result is None
    if result is None:
        print(f"Skipping circuit {number}")
        skipped_circuits += 1
        continue

    # Unpack the result
    connection_matrix, component_list = result

    # Verify matrices are not None
    if connection_matrix is None or component_list is None:
        print(f"Invalid matrix or component list for circuit {number}")
        skipped_circuits += 1
        continue

    # Save connection matrix and component list as NumPy arrays
    try:
        # Define file paths
        matrix_path = f'matrix_1/{number}.npy'
        component_list_path = f'component_lists/{number}.npy'

        # Convert Pandas DataFrame to NumPy and save
        np.save(matrix_path, connection_matrix.to_numpy())

        # Convert component list to NumPy array and save
        np.save(component_list_path, np.array(component_list))

        successful_circuits += 1
    except Exception as e:
        print(f"Error saving files for circuit {number}: {e}")
        skipped_circuits += 1

    # # Save connection matrix
    # try:
    #     csv_name = 'matrix_1/' + number + '.csv'
    #     connection_matrix.to_csv(csv_name)
        
    #     # Save component list
    #     component_list_name =  'component_list.txt'
    #     with open(component_list_name, 'a') as f:
    #         f.write('[')
    #         f.write(', '.join(map(str, component_list)))
    #         f.write(']\n')
        
    #     successful_circuits += 1
    # except Exception as e:
    #     print(f"Error saving files for circuit {number}: {e}")
    #     skipped_circuits += 1

# Print summary
print(f"\nProcessing Complete:")
print(f"Successful Circuits: {successful_circuits}")
print(f"Skipped Circuits: {skipped_circuits}")

Processing circuit 1
Directory 'Dataset/1/1.cir' does not exist. Skipping...
Processing circuit 2
Directory 'Dataset/2/2.cir' does not exist. Skipping...
Processing circuit 3
Directory 'Dataset/3/3.cir' does not exist. Skipping...
Processing circuit 4
Netlist length: 1
Ports: ['VDD', 'VSS', 'VIN1']
Unrecognized component type: nmos4
Invalid matrix or component list for circuit 4
Processing circuit 5
Directory 'Dataset/5/5.cir' does not exist. Skipping...
Processing circuit 6
Netlist length: 3
Ports: ['VDD', 'VSS', 'VOUT1']
Unrecognized component type: nmos4
Invalid matrix or component list for circuit 6
Processing circuit 7
Directory 'Dataset/7/7.cir' does not exist. Skipping...
Processing circuit 8
Directory 'Dataset/8/8.cir' does not exist. Skipping...
Processing circuit 9
Netlist length: 2
Ports: ['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1']
Unrecognized component type: pmos4
Invalid matrix or component list for circuit 9
Processing circuit 10
Directory 'Dataset/10/10.cir' does not exist. 